In [4]:
# ============================================================
# BLOQUE 0: Configuración global (imports, OMS, ventanas)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

# Ciudad y ruta base de los archivos REMMAQ
CIUDAD = "Quito"
RUTA_BASE_DATOS = os.path.join("..", "data", "remmaq", "extraidos")

# ============================================================
# Parámetros metodológicos (EDA alineado con ETL corregido)
# ============================================================
# Nota: NA/NULL en calidad del aire significa "sin medición", NO "0 mg/m3".
# Se gestiona con reglas de completitud al agregar, no imputando ceros.

DEFAULT_MIN_HORAS_DIA = 18            # >=75% de 24 horas
DEFAULT_MIN_ESTACIONES_CIUDAD = 3     # mínimo de estaciones con dato diario válido para CIUDAD

# CIUDAD:
# - "estaciones_validas": promedio solo de estaciones válidas (recomendado)
# - "den_fijo": divide por N_total estaciones (compatibilidad con lógica anterior)
DEFAULT_CIUDAD_MODO = "estaciones_validas"

# Parámetros para O3 (MDA8: máximo diario de media móvil 8h)
DEFAULT_VENTANA_8H = 8
DEFAULT_MIN_HORAS_VENTANA_8H = 6      # >=75% de 8 horas
DEFAULT_MIN_VENTANAS_DIA_8H = 18      # >=75% de ventanas del día (aprox 18/24)

# ------------------------------------------------------------
# Configuración de contaminantes y umbrales OMS (guías 2021)
# ------------------------------------------------------------
CONFIG_CONTAMINANTES = {
    "CO": {
        "archivo": "CO.xlsx",
        "unidad_datos": "mg/m3",
        "nombre_largo": "Monóxido de carbono",
        "simbolo": "CO",
        "col_fecha": "Unnamed: 0",
        "umbral_diario": {"valor": 4.0, "unidad": "mg/m3"},   # 24h
        "umbral_anual": None,
    },
    "NO2": {
        "archivo": "NO2.xlsx",
        "unidad_datos": "ug/m3",
        "nombre_largo": "Dióxido de nitrógeno",
        "simbolo": "NO₂",
        "col_fecha": "Unnamed: 0",
        "umbral_diario": {"valor": 25.0, "unidad": "ug/m3"},  # 24h
        "umbral_anual": {"valor": 10.0, "unidad": "ug/m3"},
    },
    "O3": {
    "archivo": "O3.xlsx",
    "unidad_datos": "ug/m3",
    "nombre_largo": "Ozono",
    "simbolo": "O₃",
    "col_fecha": "Unnamed: 0",

    # OMS: Métrica diaria para O3 es MDA8 (máximo diario de media móvil 8h)
    "metrica_diaria": "max_8h",
    "umbral_diario": {"valor": 100.0, "unidad": "ug/m3"},  # 8h
    "umbral_anual": None,

    # Completitud (puedes ajustarlo si justificas otro criterio)
    "min_horas_dia": DEFAULT_MIN_HORAS_DIA,
    "min_estaciones_ciudad": DEFAULT_MIN_ESTACIONES_CIUDAD,
    "ciudad_modo": DEFAULT_CIUDAD_MODO,

    "ventana_horas": DEFAULT_VENTANA_8H,
    "min_horas_ventana": DEFAULT_MIN_HORAS_VENTANA_8H,
    "min_ventanas_dia": DEFAULT_MIN_VENTANAS_DIA_8H,
    },
    "PM2.5": {
        "archivo": "PM2.5.xlsx",
        "unidad_datos": "ug/m3",
        "nombre_largo": "Material particulado fino",
        "simbolo": "PM₂.₅",
        "col_fecha": "Unnamed: 0",
        "umbral_diario": {"valor": 15.0, "unidad": "ug/m3"},
        "umbral_anual": {"valor": 5.0, "unidad": "ug/m3"},
    },
    "PM10": {
        "archivo": "PM10.xlsx",
        "unidad_datos": "ug/m3",
        "nombre_largo": "Material particulado respirable",
        "simbolo": "PM₁₀",
        "col_fecha": "Unnamed: 0",
        "umbral_diario": {"valor": 45.0, "unidad": "ug/m3"},
        "umbral_anual": {"valor": 15.0, "unidad": "ug/m3"},
    },
    "SO2": {
        "archivo": "SO2.xlsx",
        "unidad_datos": "ug/m3",
        "nombre_largo": "Dióxido de azufre",
        "simbolo": "SO₂",
        "col_fecha": "Unnamed: 0",
        "umbral_diario": {"valor": 40.0, "unidad": "ug/m3"},
        "umbral_anual": None,
    },
}

# Normalizamos la configuración para que todos los contaminantes tengan los mismos campos
for k, cfg in CONFIG_CONTAMINANTES.items():
    cfg.setdefault("metrica_diaria", "mean_24h")   # default: promedio 24h
    cfg.setdefault("min_horas_dia", DEFAULT_MIN_HORAS_DIA)
    cfg.setdefault("min_estaciones_ciudad", DEFAULT_MIN_ESTACIONES_CIUDAD)
    cfg.setdefault("ciudad_modo", DEFAULT_CIUDAD_MODO)

# Forzamos O3 a métrica OMS correcta (MDA8)
if "O3" in CONFIG_CONTAMINANTES:
    CONFIG_CONTAMINANTES["O3"]["metrica_diaria"] = "max_8h"
    CONFIG_CONTAMINANTES["O3"].setdefault("ventana_horas", DEFAULT_VENTANA_8H)
    CONFIG_CONTAMINANTES["O3"].setdefault("min_horas_ventana", DEFAULT_MIN_HORAS_VENTANA_8H)
    CONFIG_CONTAMINANTES["O3"].setdefault("min_ventanas_dia", DEFAULT_MIN_VENTANAS_DIA_8H)

# ------------------------------------------------------------
# Ventanas temporales (análisis ANUAL)
# ------------------------------------------------------------
VENTANAS_TEMPORALES = {
    "NORMALIDAD_PREVIA": {
        "nombre": "Normalidad previa",
        "anio_ini": 2015,
        "anio_fin": 2019,
        "descripcion": "Línea base sin pandemia",
    },
    "CAMBIO_MOVILIDAD": {
        "nombre": "Cambio de movilidad",
        "anio_ini": 2020,
        "anio_fin": 2021,
        "descripcion": "Régimen atípico por COVID-19",
    },
    "NUEVA_NORMALIDAD": {
        "nombre": "Nueva normalidad",
        "anio_ini": 2022,
        "anio_fin": 2025,
        "descripcion": "Base reciente para evaluación y análisis",
    },
}

def clasificar_periodo_anual(anio: int) -> str:
    if 2015 <= anio <= 2019:
        return "Normalidad previa"
    elif 2020 <= anio <= 2021:
        return "Cambio de movilidad"
    elif 2022 <= anio <= 2025:
        return "Nueva normalidad"
    else:
        return "Fuera de ventana"

ANIO_MIN_ANALISIS = min(v["anio_ini"] for v in VENTANAS_TEMPORALES.values())
ANIO_MAX_ANALISIS = max(v["anio_fin"] for v in VENTANAS_TEMPORALES.values())

print(f"Ciudad de análisis: {CIUDAD}")
print(f"Ventana global de análisis: {ANIO_MIN_ANALISIS}–{ANIO_MAX_ANALISIS}")
for v in VENTANAS_TEMPORALES.values():
    print(f"  - {v['nombre']}: {v['anio_ini']}–{v['anio_fin']} | {v['descripcion']}")

Ciudad de análisis: Quito
Ventana global de análisis: 2015–2025
  - Normalidad previa: 2015–2019 | Línea base sin pandemia
  - Cambio de movilidad: 2020–2021 | Régimen atípico por COVID-19
  - Nueva normalidad: 2022–2025 | Base reciente para evaluación y análisis


In [ ]:
# ============================================================
# BLOQUE 1: Carga y limpieza básica (DATASET LIMPIO horario)
# ============================================================

def cargar_y_limpiar_contaminante(nombre_contaminante: str):
    """
    Lee el Excel del contaminante (formato nuevo o antiguo),
    estandariza la columna FECHA, elimina filas inválidas,
    convierte estaciones a numérico y devuelve un dataset horario limpio.

    Formato NUEVO (el que muestras en la segunda imagen):
        Encabezado:  FECHA / Fecha, BELISARIO, CARAPUNGO, ...
        Fila 2+:     datos horarios

    Formato ANTIGUO:
        Fila 1:      'FECHA \\ UNIDAD', 'ug/m3', ...
        Fila 2+:     datos horarios
    """
    cfg = CONFIG_CONTAMINANTES[nombre_contaminante]
    ruta_archivo = os.path.join(RUTA_BASE_DATOS, cfg["archivo"])

    print(f"\n=== Cargando {nombre_contaminante} desde: {ruta_archivo} ===")
    df_raw = pd.read_excel(ruta_archivo)
    print(f"[{nombre_contaminante}] df_raw.shape = {df_raw.shape}")
    print(f"[{nombre_contaminante}] Columnas originales: {list(df_raw.columns)}")

    # -------------------------------------------------------
    # 1. Detectar columna de fecha por nombre (case-insensitive)
    # -------------------------------------------------------
    col_fecha = None

    # 1.a Si en la config hay col_fecha y existe, úsala
    col_fecha_cfg = cfg.get("col_fecha")
    if col_fecha_cfg and col_fecha_cfg in df_raw.columns:
        col_fecha = col_fecha_cfg

    # 1.b Si no, busca cualquier columna cuyo nombre contenga 'FECHA' (cubre 'Fecha', 'FECHA', etc.)
    if col_fecha is None:
        for c in df_raw.columns:
            if "FECHA" in str(c).upper():
                col_fecha = c
                break

    if col_fecha is None:
        raise ValueError(
            f"[{nombre_contaminante}] No se encontró columna de fecha. "
            f"Columnas = {list(df_raw.columns)}"
        )

    print(f"[{nombre_contaminante}] Columna de fecha detectada: {col_fecha!r}")

    # Renombramos esa columna a 'FECHA' para estandarizar
    df_raw = df_raw.rename(columns={col_fecha: "FECHA"})
    col_fecha = "FECHA"

    # -------------------------------------------------------
    # 2. Detectar si hay fila de unidades (formato antiguo)
    #    Miramos el valor de la PRIMERA FILA en la columna FECHA
    # -------------------------------------------------------
    valor_fila0 = str(df_raw[col_fecha].iloc[0]).upper()

    if "FECHA" in valor_fila0:  # típico 'FECHA \\ UNIDAD'
        # ---------- FORMATO ANTIGUO ----------
        print(f"[{nombre_contaminante}] Formato ANTIGUO (fila de unidades) detectado")

        unidades = df_raw.iloc[0].copy()
        df_num = df_raw.iloc[1:].copy()   # datos sin la fila de unidades
    else:
        # ---------- FORMATO NUEVO ----------
        print(f"[{nombre_contaminante}] Formato NUEVO (una fila de encabezados) detectado")

        # No tenemos fila de unidades; las derivamos desde la configuración
        unidades = pd.Series(
            {c: (cfg["unidad_datos"] if c != "FECHA" else "fecha")
             for c in df_raw.columns},
            name="UNIDAD"
        )
        df_num = df_raw.copy()

    # -------------------------------------------------------
    # 3. Conversión de FECHA y filtrado por ventana temporal
    # -------------------------------------------------------
    df_num["FECHA"] = pd.to_datetime(df_num["FECHA"], errors="coerce")
    df_num = df_num.dropna(subset=["FECHA"])

    df_num = df_num[
        (df_num["FECHA"].dt.year >= ANIO_MIN_ANALISIS)
        & (df_num["FECHA"].dt.year <= ANIO_MAX_ANALISIS)
    ].copy()

    # -------------------------------------------------------
    # 4. Estaciones y conversión a numérico (SIN imputar 0)
    # -------------------------------------------------------
    columnas_estaciones = [c for c in df_num.columns if c != "FECHA"]

    # df_num[columnas_estaciones] = df_num[columnas_estaciones].apply(
    #     pd.to_numeric, errors="coerce"
    # )
    # df_num[columnas_estaciones] = df_num[columnas_estaciones].fillna(0)

    df_num[columnas_estaciones] = df_num[columnas_estaciones].apply(
    pd.to_numeric, errors="coerce"
    )

    # IMPORTANTE (TFM):
    # No imputamos NaN -> 0. "Sin medición" no equivale a 0.
    # Los faltantes se gestionan con reglas de completitud al calcular agregados diarios.


    print(f"[{nombre_contaminante}] df_horario limpio shape = {df_num.shape}")
    print(f"[{nombre_contaminante}] Estaciones detectadas: {columnas_estaciones}")

    return {
        "raw": df_raw,
        "unidades": unidades,
        "df_horario": df_num,
        "columnas_estaciones": columnas_estaciones,
    }

# Cargar todos los contaminantes con la nueva función robusta
datos_contaminantes = {}
for nombre in CONFIG_CONTAMINANTES.keys():
    datos_contaminantes[nombre] = cargar_y_limpiar_contaminante(nombre)


In [ ]:
# ============================================================
# BLOQUE 2: Enriquecimiento horario (ANIO, PERIODO, FECHA_DIA)
# ============================================================

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 2: Enriquecimiento horario {nombre} ({CIUDAD}) ===")

    df_h = info["df_horario"].copy()

    df_h["ANIO"] = df_h["FECHA"].dt.year.astype("Int64")
    df_h["PERIODO"] = df_h["ANIO"].apply(clasificar_periodo_anual)
    df_h["FECHA_DIA"] = df_h["FECHA"].dt.floor("D")

    info["df_horario_enriquecido"] = df_h

    print(f"[{nombre}] df_horario_enriquecido.shape = {df_h.shape}")
    print("\nDistribución por PERIODO (dataset limpio horario):")
    display(df_h["PERIODO"].value_counts(dropna=False))

In [ ]:
# ============================================================
# BLOQUE 3: Agregación diaria y anual (con completitud + O3 MDA8)
# ============================================================

def agregar_diario_por_contaminante(df_h: pd.DataFrame, cols_est: list[str], cfg: dict) -> pd.DataFrame:
    """
    Devuelve df_diario con:
      - valores diarios por estación (según metrica_diaria)
      - CIUDAD diaria (promedio espacial)
      - ESTACIONES_VALIDAS_DIA (auditoría)
      - ANIO y PERIODO

    Reglas clave:
      - NA/NULL se interpreta como "sin medición" (no se convierte a 0).
      - Se exige completitud mínima para publicar valores diarios.
    """
    metrica = cfg.get("metrica_diaria", "mean_24h")
    min_horas_dia = int(cfg.get("min_horas_dia", DEFAULT_MIN_HORAS_DIA))
    min_est_ciudad = int(cfg.get("min_estaciones_ciudad", DEFAULT_MIN_ESTACIONES_CIUDAD))
    ciudad_modo = cfg.get("ciudad_modo", DEFAULT_CIUDAD_MODO)

    # Asegurar orden temporal
    df_h = df_h.sort_values("FECHA").copy()

    if metrica == "mean_24h":
        # Promedio diario 24h por estación (ignora NaN)
        mean_d = df_h.groupby(["FECHA_DIA", "ANIO", "PERIODO"])[cols_est].mean()

        # Horas válidas por estación/día
        count_h = df_h.groupby(["FECHA_DIA", "ANIO", "PERIODO"])[cols_est].count()

        df_d = mean_d.reset_index()

        # Completitud: si faltan muchas horas, no publicamos el valor diario
        for c in cols_est:
            df_d.loc[count_h[c].values < min_horas_dia, c] = np.nan

    elif metrica == "max_8h":
        # O3 (OMS): máximo diario de media móvil 8h (MDA8)
        ventana_horas = int(cfg.get("ventana_horas", DEFAULT_VENTANA_8H))
        min_horas_ventana = int(cfg.get("min_horas_ventana", DEFAULT_MIN_HORAS_VENTANA_8H))
        min_ventanas_dia = int(cfg.get("min_ventanas_dia", DEFAULT_MIN_VENTANAS_DIA_8H))

        # Rolling 8h sobre índice horario real
        df_idx = df_h.set_index("FECHA")[cols_est]
        roll = df_idx.rolling(f"{ventana_horas}h", min_periods=min_horas_ventana).mean().reset_index()

        roll["FECHA_DIA"] = roll["FECHA"].dt.floor("D")

        # Máximo de la media móvil 8h por día (día del final de la ventana)
        max_8h = roll.groupby("FECHA_DIA")[cols_est].max()

        # Conteo de ventanas calculables por día
        cnt_win = roll.groupby("FECHA_DIA")[cols_est].count()

        # Conteo de horas válidas por día (para asegurar cobertura diaria también)
        cnt_hr = df_h.groupby("FECHA_DIA")[cols_est].count()

        df_d = max_8h.reset_index()

        # Reconstruir ANIO/PERIODO desde FECHA_DIA
        df_d["ANIO"] = df_d["FECHA_DIA"].dt.year.astype("Int64")
        df_d["PERIODO"] = df_d["ANIO"].apply(clasificar_periodo_anual)

        # Completitud doble: horas válidas del día y suficientes ventanas MDA8
        for c in cols_est:
            df_d.loc[cnt_hr[c].reindex(df_d["FECHA_DIA"]).values < min_horas_dia, c] = np.nan
            df_d.loc[cnt_win[c].reindex(df_d["FECHA_DIA"]).values < min_ventanas_dia, c] = np.nan

        # Reordenar columnas para consistencia con otros contaminantes
        df_d = df_d[["FECHA_DIA", "ANIO", "PERIODO"] + cols_est]

    else:
        raise ValueError(f"Métrica diaria desconocida: {metrica}")

    # CIUDAD diaria + auditoría de estaciones válidas
    n_est_validas = df_d[cols_est].notna().sum(axis=1)
    df_d["ESTACIONES_VALIDAS_DIA"] = n_est_validas.astype("Int64")

    if ciudad_modo == "den_fijo":
        # Compatibilidad: faltantes tratados como 0 SOLO para CIUDAD y divisor fijo por N estaciones
        df_d["CIUDAD"] = df_d[cols_est].fillna(0).sum(axis=1) / float(len(cols_est))
    else:
        # Recomendado: promedio solo entre estaciones con valor diario válido
        df_d["CIUDAD"] = df_d[cols_est].mean(axis=1, skipna=True)

    # Representatividad mínima
    df_d.loc[n_est_validas < min_est_ciudad, "CIUDAD"] = np.nan

    # Redondeo solo para presentación (conserva NaN)
    df_d[cols_est + ["CIUDAD"]] = df_d[cols_est + ["CIUDAD"]].round(2)

    return df_d


for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 3: Agregación diaria/anual {nombre} ({CIUDAD}) ===")

    cfg = CONFIG_CONTAMINANTES[nombre]
    df_h = info["df_horario_enriquecido"].copy()
    cols_est = info["columnas_estaciones"]

    # Restringimos a ventana global de análisis (a nivel horario)
    df_h = df_h[
        (df_h["ANIO"] >= ANIO_MIN_ANALISIS) & (df_h["ANIO"] <= ANIO_MAX_ANALISIS)
    ].copy()

    # 3.1 Diario (según métrica del contaminante + completitud)
    df_diario = agregar_diario_por_contaminante(df_h, cols_est, cfg)

    print(f"[{nombre}] df_diario.shape = {df_diario.shape}")
    display(df_diario.head())

    # 3.2 LONG diario (generalizado: no asumir 24h porque O3 es MDA8)
    df_diario_long = df_diario.melt(
        id_vars=["FECHA_DIA", "ANIO", "PERIODO"],
        value_vars=cols_est,
        var_name="ESTACION",
        value_name="CONCENTRACION_DIARIA",
    )
    df_diario_long["CONTAMINANTE"] = nombre
    df_diario_long["UNIDAD"] = cfg["unidad_datos"]
    df_diario_long["METRICA_DIARIA"] = cfg.get("metrica_diaria", "mean_24h")

    # 3.3 Media anual por estación (sobre la métrica diaria disponible)
    df_anual_est_long = (
        df_diario_long
        .groupby(["ANIO", "PERIODO", "ESTACION"])["CONCENTRACION_DIARIA"]
        .mean()
        .round(2)
        .reset_index()
        .rename(columns={"CONCENTRACION_DIARIA": "PROM_ANUAL"})
    )
    df_anual_est_long["CONTAMINANTE"] = nombre

    df_anual_est_wide = (
        df_anual_est_long
        .pivot_table(index="ANIO", columns="ESTACION", values="PROM_ANUAL")
        .round(2)
    )
    df_anual_est_wide["PERIODO"] = df_anual_est_wide.index.to_series().apply(clasificar_periodo_anual)

    # 3.4 Media anual ciudad + MAX_DIARIO (máximo de la serie diaria CIUDAD del año)
    df_anual_ciudad = (
        df_diario
        .groupby(["ANIO", "PERIODO"])["CIUDAD"]
        .agg(PROM_ANUAL="mean", MAX_DIARIO="max")
        .round(2)
        .reset_index()
    )
    df_anual_ciudad["CONTAMINANTE"] = nombre
    df_anual_ciudad["METRICA_DIARIA"] = cfg.get("metrica_diaria", "mean_24h")

    # Guardar en dict
    info["df_diario"] = df_diario
    info["df_diario_long"] = df_diario_long
    info["anual_estaciones"] = df_anual_est_wide
    info["anual_estaciones_long"] = df_anual_est_long
    info["anual_ciudad"] = df_anual_ciudad


In [ ]:
# ============================================================
# BLOQUE 4: Calidad de datos (faltantes y ceros reales)
# ============================================================

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 4: Calidad de datos {nombre} ({CIUDAD}) ===")

    cfg = CONFIG_CONTAMINANTES[nombre]
    cols_est = info["columnas_estaciones"]

    # Dataset horario (con NaN reales)
    df_h = info["df_horario_enriquecido"].copy()
    df_h = df_h[
        (df_h["ANIO"] >= ANIO_MIN_ANALISIS) & (df_h["ANIO"] <= ANIO_MAX_ANALISIS)
    ].copy()

    # 4.1 % faltantes horario (por estación)
    pct_faltantes_global = (df_h[cols_est].isna().mean() * 100).round(2)

    # 4.2 % ceros horario SOLO entre observaciones válidas (no confundir con faltantes)
    ceros = df_h[cols_est].eq(0).sum()
    obs = df_h[cols_est].notna().sum()
    pct_ceros_obs_global = (ceros / obs * 100).replace([np.inf, -np.inf], np.nan).round(2)

    print("\n% faltantes (NaN) por estación (horario):")
    display(pct_faltantes_global.sort_values(ascending=False))

    print("\n% ceros por estación (horario, solo sobre valores observados):")
    display(pct_ceros_obs_global.sort_values(ascending=False))

    # 4.3 Evolución anual (% faltantes y % ceros sobre observados)
    pct_faltantes_anual = (
        df_h.groupby("ANIO")[cols_est]
        .apply(lambda g: g.isna().mean() * 100)
        .round(2)
    )

    pct_ceros_anual_obs = (
        df_h.groupby("ANIO")[cols_est]
        .apply(lambda g: (g.eq(0).sum() / g.notna().sum()) * 100)
        .replace([np.inf, -np.inf], np.nan)
        .round(2)
    )

    info["pct_faltantes_horario_global"] = pct_faltantes_global
    info["pct_ceros_horario_global_obs"] = pct_ceros_obs_global
    info["pct_faltantes_horario_anual"] = pct_faltantes_anual
    info["pct_ceros_horario_anual_obs"] = pct_ceros_anual_obs

    # 4.4 Calidad diaria (post-completitud): días válidos vs esperados
    df_d = info["df_diario"].copy()

    # Días esperados por año = días con al menos 1 registro horario (por fecha)
    dias_esperados = df_h.groupby("ANIO")["FECHA_DIA"].nunique().rename("DIAS_ESPERADOS")

    # Días válidos por estación = count() ignora NaN (ya aplica reglas de completitud)
    dias_validos_est = df_d.groupby("ANIO")[cols_est].count()
    dias_validos_ciudad = df_d.groupby("ANIO")["CIUDAD"].count().rename("DIAS_VALIDOS_CIUDAD")

    # Resumen por estación (long)
    calidad_est = (
        dias_validos_est
        .stack()
        .rename("DIAS_VALIDOS")
        .reset_index()
        .rename(columns={"level_1": "ESTACION"})
        .merge(dias_esperados.reset_index(), on="ANIO", how="left")
    )
    calidad_est["DIAS_FALTANTES"] = (calidad_est["DIAS_ESPERADOS"] - calidad_est["DIAS_VALIDOS"]).clip(lower=0)
    calidad_est["PCT_DIAS_VALIDOS"] = (calidad_est["DIAS_VALIDOS"] / calidad_est["DIAS_ESPERADOS"] * 100).round(1)
    calidad_est["PERIODO"] = calidad_est["ANIO"].apply(clasificar_periodo_anual)
    calidad_est["CONTAMINANTE"] = nombre

    # Resumen ciudad
    calidad_ciudad = (
        dias_validos_ciudad.reset_index()
        .merge(dias_esperados.reset_index(), on="ANIO", how="left")
    )
    calidad_ciudad["DIAS_FALTANTES"] = (calidad_ciudad["DIAS_ESPERADOS"] - calidad_ciudad["DIAS_VALIDOS_CIUDAD"]).clip(lower=0)
    calidad_ciudad["PCT_DIAS_VALIDOS"] = (calidad_ciudad["DIAS_VALIDOS_CIUDAD"] / calidad_ciudad["DIAS_ESPERADOS"] * 100).round(1)
    calidad_ciudad["PERIODO"] = calidad_ciudad["ANIO"].apply(clasificar_periodo_anual)
    calidad_ciudad["CONTAMINANTE"] = nombre
    calidad_ciudad["METRICA_DIARIA"] = cfg.get("metrica_diaria", "mean_24h")

    info["calidad_estaciones"] = calidad_est
    info["calidad_ciudad"] = calidad_ciudad

    print(f"[{nombre}] calidad_estaciones.shape = {calidad_est.shape}")
    print(f"[{nombre}] calidad_ciudad.shape = {calidad_ciudad.shape}")
    display(calidad_ciudad.head())

In [ ]:
# ============================================================
# BLOQUE 5: Excedencias de la guía OMS (diario y anual)
# ============================================================

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 5: Excedencias OMS {nombre} ({CIUDAD}) ===")

    cfg = CONFIG_CONTAMINANTES[nombre]
    df_d = info["df_diario"].copy()

    umbral_diario = cfg["umbral_diario"]["valor"]
    metrica = cfg.get("metrica_diaria", "mean_24h")

    # 5.1 Bandera diaria para CIUDAD (usa CIUDAD ya calculado según métrica del contaminante)
    df_d["EXCEDE_OMS_DIARIO_CIUDAD"] = df_d["CIUDAD"] > umbral_diario

    exc_ciudad = (
        df_d.groupby(["ANIO", "PERIODO"])
        .agg(
            DIAS_TOTALES=("CIUDAD", "count"),  # count ignora NaN -> solo días válidos
            DIAS_EXC=("EXCEDE_OMS_DIARIO_CIUDAD", "sum"),
        )
        .reset_index()
    )
    exc_ciudad["PCT_DIAS_EXC"] = (exc_ciudad["DIAS_EXC"] / exc_ciudad["DIAS_TOTALES"] * 100).round(1)
    exc_ciudad["UMBRAL_DIARIO"] = umbral_diario
    exc_ciudad["CONTAMINANTE"] = nombre
    exc_ciudad["METRICA_DIARIA"] = metrica

    print("\n% de días válidos por año con CIUDAD > guía OMS (según métrica diaria):")
    display(exc_ciudad.head())

    # 5.2 Estaciones: usamos df_diario_long (generalizado)
    df_long = info["df_diario_long"].copy()
    df_long["EXCEDE_OMS_DIARIO"] = df_long["CONCENTRACION_DIARIA"] > umbral_diario

    exc_est = (
        df_long.groupby(["ANIO", "PERIODO", "ESTACION"])
        .agg(
            DIAS_TOTALES=("CONCENTRACION_DIARIA", "count"),
            DIAS_EXC=("EXCEDE_OMS_DIARIO", "sum"),
        )
        .reset_index()
    )
    exc_est["PCT_DIAS_EXC"] = (exc_est["DIAS_EXC"] / exc_est["DIAS_TOTALES"] * 100).round(1)
    exc_est["UMBRAL_DIARIO"] = umbral_diario
    exc_est["CONTAMINANTE"] = nombre
    exc_est["METRICA_DIARIA"] = metrica

    print("\n% de días válidos por año con estación > guía OMS (según métrica diaria):")
    display(exc_est.head())

    info["excedencias_diarias_ciudad"] = exc_ciudad
    info["excedencias_diarias_estaciones"] = exc_est

    # 5.3 Excedencias anuales (si existe guía anual OMS)
    umbral_anual_cfg = cfg["umbral_anual"]
    if umbral_anual_cfg is not None:
        umbral_anual = umbral_anual_cfg["valor"]

        df_an_ciudad = info["anual_ciudad"].copy()
        df_an_ciudad["UMBRAL_ANUAL"] = umbral_anual
        df_an_ciudad["SUPERA_OMS_ANUAL"] = df_an_ciudad["PROM_ANUAL"] > umbral_anual
        df_an_ciudad["APLICA_UMBRAL_ANUAL"] = True

        df_an_est = info["anual_estaciones_long"].copy()
        df_an_est["UMBRAL_ANUAL"] = umbral_anual
        df_an_est["SUPERA_OMS_ANUAL"] = df_an_est["PROM_ANUAL"] > umbral_anual
        df_an_est["APLICA_UMBRAL_ANUAL"] = True

        info["excedencias_anuales_ciudad"] = df_an_ciudad
        info["excedencias_anuales_estaciones"] = df_an_est
    else:
        info["excedencias_anuales_ciudad"] = None
        info["excedencias_anuales_estaciones"] = None

In [ ]:
# ============================================================
# BLOQUE 6: Detección de outliers (IQR) sobre datos diarios
# ============================================================

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 6: Outliers {nombre} ({CIUDAD}) ===")

    df_long = info["df_diario_long"].copy()

    stats_out = df_long.groupby("ESTACION")["CONCENTRACION_DIARIA"].describe()
    stats_out["IQR"] = stats_out["75%"] - stats_out["25%"]
    stats_out["UMBRAL_ALTO"] = stats_out["75%"] + 1.5 * stats_out["IQR"]

    print("\nEstadísticos por estación para outliers (IQR):")
    display(stats_out[["min", "25%", "50%", "75%", "max", "UMBRAL_ALTO"]])

    outliers_reg = df_long.merge(
        stats_out["UMBRAL_ALTO"],
        left_on="ESTACION",
        right_index=True
    )

    outliers_reg = outliers_reg[
        outliers_reg["CONCENTRACION_DIARIA"] > outliers_reg["UMBRAL_ALTO"]
    ].sort_values("CONCENTRACION_DIARIA", ascending=False)

    info["outliers_stats"] = stats_out
    info["outliers_registros"] = outliers_reg

    print(f"[{nombre}] outliers_registros.shape = {outliers_reg.shape}")
    display(outliers_reg.head(20))


In [ ]:
# ============================================================
# BLOQUE 7: Preparación para gráficas (ventanas + truncado)
# ============================================================

periodos_validos = [v["nombre"] for v in VENTANAS_TEMPORALES.values()]

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 7: Preparación gráficas {nombre} ({CIUDAD}) ===")

    ciudad = info["anual_ciudad"].copy()

    mask_ciudad = (
        ciudad["ANIO"].between(ANIO_MIN_ANALISIS, ANIO_MAX_ANALISIS)
        & ciudad["PERIODO"].isin(periodos_validos)
    )
    ciudad_analisis = ciudad[mask_ciudad].copy()

    if ciudad_analisis.empty:
        print("Sin datos en la ventana de análisis para ciudad.")
        info["ciudad_analisis_vis"] = ciudad_analisis
        info["estaciones_analisis_vis"] = info["anual_estaciones_long"].iloc[0:0]
        continue

    p95_ciudad = ciudad_analisis["PROM_ANUAL"].quantile(0.95)
    ciudad_analisis["PROM_VIS"] = ciudad_analisis["PROM_ANUAL"].clip(upper=p95_ciudad)

    est_long = info["anual_estaciones_long"].copy()
    mask_est = (
        est_long["ANIO"].between(ANIO_MIN_ANALISIS, ANIO_MAX_ANALISIS)
        & est_long["PERIODO"].isin(periodos_validos)
    )
    est_analisis = est_long[mask_est].copy()

    p95_est = est_analisis["PROM_ANUAL"].quantile(0.95)
    est_analisis["PROM_VIS"] = est_analisis["PROM_ANUAL"].clip(upper=p95_est)

    resumen_ciudad = (
        ciudad_analisis
        .groupby("PERIODO")["PROM_ANUAL"]
        .mean()
        .round(3)
        .reset_index(name="PROM_ANUAL_MEDIO")
    )

    resumen_est = (
        est_analisis
        .groupby(["ESTACION", "PERIODO"])["PROM_ANUAL"]
        .mean()
        .round(3)
        .reset_index()
    )

    info["ciudad_analisis_vis"] = ciudad_analisis
    info["estaciones_analisis_vis"] = est_analisis
    info["resumen_ventanas_ciudad"] = resumen_ciudad
    info["resumen_ventanas_estacion"] = resumen_est

    print("\nResumen de ciudad por ventana temporal:")
    display(resumen_ciudad.head())

    print("\nResumen por estación y ventana temporal:")
    display(resumen_est.head())


In [ ]:
# ============================================================
# BLOQUE 8: Gráficas de ciudad (anual + ventanas)
# ============================================================

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 8: Gráficas ciudad {nombre} ({CIUDAD}) ===")

    cfg = CONFIG_CONTAMINANTES[nombre]
    ciudad_vis = info["ciudad_analisis_vis"].copy()

    if ciudad_vis.empty:
        print("Sin datos para graficar ciudad.")
        continue

    # Umbral de referencia
    if cfg["umbral_anual"] is not None:
        umbral_ref = cfg["umbral_anual"]["valor"]
        tipo_umbral = "anual"
    else:
        umbral_ref = cfg["umbral_diario"]["valor"]
        tipo_umbral = "24h"

    unidad = cfg["unidad_datos"]

    plt.figure(figsize=(10, 5))
    ax = sns.barplot(
        data=ciudad_vis,
        x="ANIO",
        y="PROM_VIS",
        hue="PERIODO",
        dodge=False,
        order=sorted(ciudad_vis["ANIO"].unique())
    )

    plt.axhline(
        umbral_ref,
        linestyle="--",
        color="red",
        label=f"Guía OMS {tipo_umbral} ({umbral_ref:.1f} {unidad})"
    )

    plt.title(f"{nombre}: promedio anual de {CIUDAD} (dataset limpio)")
    plt.xlabel("Año")
    plt.ylabel(f"Concentración promedio anual ({unidad})")

    for p in ax.patches:
        valor = p.get_height()
        if valor > 0:
            ax.annotate(
                f"{valor:.2f}",
                (p.get_x() + p.get_width()/2, valor),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8
            )

    plt.legend(loc="upper right", title="Periodo")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# BLOQUE 9: Gráficas por estación (promedio anual)
# ============================================================

for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 9: Gráficas estaciones {nombre} ({CIUDAD}) ===")

    cfg = CONFIG_CONTAMINANTES[nombre]
    est_vis = info["estaciones_analisis_vis"].copy()

    if est_vis.empty:
        print("Sin datos para graficar estaciones.")
        continue

    unidad = cfg["unidad_datos"]

    if cfg["umbral_anual"] is not None:
        umbral_ref = cfg["umbral_anual"]["valor"]
        tipo_umbral = "anual"
    else:
        umbral_ref = cfg["umbral_diario"]["valor"]
        tipo_umbral = "24h"

    plt.figure(figsize=(12, 6))
    ax = sns.lineplot(
        data=est_vis,
        x="ANIO",
        y="PROM_VIS",
        hue="ESTACION",
        marker="o"
    )

    plt.axhline(
        umbral_ref,
        linestyle="--",
        color="red",
        label=f"Guía OMS {tipo_umbral} ({umbral_ref:.1f} {unidad})"
    )

    plt.title(f"{nombre}: promedio anual por estación (dataset limpio)")
    plt.xlabel("Año")
    plt.ylabel(f"Concentración promedio anual ({unidad})")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# BLOQUE 10: Correlación entre estaciones por ventana temporal
# ============================================================
for nombre, info in datos_contaminantes.items():
    print(f"\n=== BLOQUE 10: Correlaciones {nombre} ({CIUDAD}) ===")

    cfg = CONFIG_CONTAMINANTES[nombre]
    df_d = info["df_diario"].copy()
    cols_est = info["columnas_estaciones"]

    # Estaciones "buenas": menos del 40% de faltantes horario (no confundir con ceros reales)
    pct_faltantes_global = info.get("pct_faltantes_horario_global")
    if pct_faltantes_global is None:
        print("No existe pct_faltantes_horario_global. Ejecuta primero el BLOQUE 4.")
        continue

    estaciones_buenas = pct_faltantes_global[pct_faltantes_global < 40].index.tolist()

    if len(estaciones_buenas) < 2:
        print("No hay suficientes estaciones con cobertura para correlación.")
        continue

    # Correlación sobre datos diarios, solo con estaciones buenas
    df_corr = df_d[estaciones_buenas].copy()

    # Para correlación, eliminamos días donde faltan todas las estaciones buenas
    df_corr = df_corr.dropna(how="all")

    corr = df_corr.corr()

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
    plt.title(f"{nombre}: correlación diaria entre estaciones\nMétrica: {cfg.get('metrica_diaria','mean_24h')}")
    plt.tight_layout()
    plt.show()

    info["correlaciones_estaciones"] = corr